### Essentially, fine-tuning embedding models on your data to improve your RAG application!

the most efficient way to get this improvement is through a query-only linear adapter, or training a simple linear layer transformation to better represent user queries in embedding space for improved retrieval.

This allows us to very easily plug into existing RAG pipelines and optimize for our specific task without needing to completely re-embed our knowledge base or use a lot of resources training larger models, making this a simple, cost/compute-effective way to improve retrieval performance.


![](adapters_explainer.png)

In this notebook we will:

1. Define a RAG application to optimize
2. Generate a synthetic dataset with gpt-4o-mini
3. Test retrieval metrics to gather a baseline for all-MiniLM-L6-v2
4. Create and train a linear adapter
5. Plug the adapter onto all-MiniLM-L6-v2 and assess performance

we'll be implementing many methodologies from ChromaDB's Research on a small scale for task-specific performance increases, specifically their recommendations for triplet loss, random negative sampling, and linear query-only transformation.

### Synth data creation

To optimize document retrieval effectively, a crucial component is having access to high-quality labeled data. This data typically consists of pairs matching user queries with their most relevant documents. While the ideal scenario would involve collecting and labeling this data from real-world user interactions and testing, for demonstration purposes we can simulate this data by generating potential queries for each chunk of our knowledgebase.

As a plus [it's been researched](https://arxiv.org/pdf/2401.00368) that using LLMs to generate synthetic data for text embedding improvement can lead to gains!

### Loading [Apple's 2024 Environmental Report](https://www.apple.com/environment/pdf/Apple_Environmental_Progress_Report_2024.pdf)

In [22]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the project API configuration without printing secrets.
load_dotenv('../../.env')

# Will be used as Training Data
apple_loader = PyPDFLoader("Apple_Environmental_Progress_Report_2024.pdf")
apple_pages = apple_loader.load()

apple_document = ""
for i in range(len(apple_pages)):
    apple_document += apple_pages[i].page_content

In [2]:
# Split PDFs
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4",
    chunk_size=800,
    chunk_overlap=400,
)

apple_chunks = text_splitter.split_text(apple_document)

In [23]:
apple_chunks

['Environment al \nProgress \nReport\nCovering fiscal year 2023Contents\nEnvironmental \nInitiatives\nApple 2030\n 11 Journey to Apple 2030\n 12 Approach\n 15 Design and materials\n 24 Electricity\n 32 Direct emissions\n 35 Carbon removal\nResources\n 39 Approach\n 40 Product longevity\n 45 Material recovery\n 48 Water\n 52 Zero waste\nSmarter Chemistry\n 58 Approach\n 59 Mapping\n 61 Assessment\n 63 InnovationIntroduction\n 3 Reflections from Lisa Jackson\n 4 Report highlights\n 5 Goals  and progressEngagement and \nAdvocacy\n 67 Approach\n 68 Listening to a range of voices\n 69 Achieving change together\n 73 Supporting communities \nworldwide\nData\n 77 Greenhouse gas emissions\n 78 High quality carbon certificates\n 79 Carbon footprint by product\n 81 Energy\n 82 Resources\n 83 Normalizing factorsAppendix\n 85 A : Corporate facilities energy supplement\n 94 B: Apple’s life cycle assessment methodology\n 96 C: Assurance and review statements\n 107 D : Environment, Health and Safety P

for demonstration we will use some synthetic QA pair generation via an LLM.

The hope is that for each chunk of text that we have, we can create a possible user query that would most likely retrieve that chunk of text. This will allow us to further on down the line test the same user query, and assess based on retrieval/rank of the expected chunk.

In [3]:
label_template = """
You are an AI assistant tasked with generating a single, realistic question-answer pair based on a given document. The question should be something a user might naturally ask when seeking information contained in the document.

Given: {chunk}

Instructions:
1. Analyze the key topics, facts, and concepts in the given document, choose one to focus on.
2. Generate twenty similar questions that a user might ask to find the information in this document that does NOT contain any company name.
3. Use natural language and occasionally include typos or colloquialisms to mimic real user behavior in the question.
4. Ensure the question is semantically related to the document content WITHOUT directly copying phrases.
5. Make sure that all of the questions are similar to eachother. I.E. All asking about a similar topic/requesting the same information.

Output Format:
Return a JSON object with the following structure:
```json
{{
  "question_1": "Generated question text",
  "question_2": "Generated question text",
  ...
}}
```

Be creative, think like a curious user, and generate your 20 similar questions that would naturally lead to the given document in a semantic search. Ensure your response is a valid JSON object containing only the questions.

"""

label_prompt = ChatPromptTemplate.from_template(label_template)
llm = ChatOpenAI(temperature=1.0, model="gpt-4o-mini")

label_chain = label_prompt | llm | JsonOutputParser()

In [4]:
# The supplied pre-generated labels in data/apple_QA_dataset.json are used below.
# To generate a new example with the configured OpenAI key, uncomment the next line.
# label_chain.invoke({"chunk": apple_chunks[20]})

save the chuncks as:
- Training set size: 3440
- Validation set size: 860

In [5]:
# Expand the supplied 20 questions per document chunk, then make a reproducible 80/20 split.
# This produces 3,440 training and 860 validation examples from the 215 source chunks.
import random

with open('data/apple_QA_dataset.json', 'r') as f:
    qa_records = json.load(f)

# Index the canonical chunks associated with the supplied questions.  PDF text
# extraction can vary between library versions, so these strings are the
# reliable ground truth for the exact-match retrieval metrics below.
apple_chunks = [record['input_chunk'] for record in qa_records]

# Split at the chunk level so validation questions come from unseen documents.
# Splitting individual questions would leak each document into both sets.
shuffled_records = qa_records.copy()
random.Random(42).shuffle(shuffled_records)
split_index = int(len(shuffled_records) * 0.8)
train_records = shuffled_records[:split_index]
validation_records = shuffled_records[split_index:]

def expand_questions(records):
    return [
        {'question': question, 'chunk': record['input_chunk']}
        for record in records
        for question in record['questions'].values()
    ]

all_data = [
    {'question': question, 'chunk': record['input_chunk']}
    for record in qa_records
    for question in record['questions'].values()
]
train_data = expand_questions(train_records)
validation_data = expand_questions(validation_records)

with open('data/train.json', 'w') as f:
    json.dump(train_data, f, indent=2)
with open('data/validation.json', 'w') as f:
    json.dump(validation_data, f, indent=2)

print(f'Training examples: {len(train_data)}; validation examples: {len(validation_data)}')

Training examples: 3440; validation examples: 860


In [6]:
import chromadb

# Create a local persistent Chroma client.
client = chromadb.PersistentClient(path='chromadb')

# Create collections for both our specific simulated RAG pipelines
apple_collection = client.get_or_create_collection(name='apple_collection', metadata={"hnsw:space": "cosine"})

In [7]:
# Embed the corpus with the same model used for queries.  Supplying the
# embeddings explicitly avoids mixing Chroma's default ONNX embedder with
# SentenceTransformer query embeddings.
from sentence_transformers import SentenceTransformer

base_model = SentenceTransformer('all-MiniLM-L6-v2')
apple_collection.upsert(
    documents=apple_chunks,
    embeddings=base_model.encode(apple_chunks).tolist(),
    ids=[f'chunk_{i}' for i in range(len(apple_chunks))],
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
def retrieve_documents_embeddings(query_embedding, k=10):
    query_embedding_list = query_embedding.tolist()
    
    results = apple_collection.query(
        query_embeddings=[query_embedding_list],
        n_results=k)
    return results['documents'][0]

### Metric: Mean Reciprocal Rank (MRR)

$$MRR = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{rank_i}$$

Where:

- $|Q|$ is the number of queries
- $rank_i$ is the rank of the first correct answer for the $i$-th query

MRR (Mean Reciprocal Rank) measures how high the first correct answer appears in the list, on average. MRR is particularly useful when there's only one relevant item in the list or when we're primarily interested in the position of the first correct result

In [9]:
def reciprocal_rank(retrieved_docs, ground_truth, k):
    try:
        rank = retrieved_docs.index(ground_truth) + 1
        return 1.0 / rank if rank <= k else 0.0
    except ValueError:
        return 0.0

### Metric: Recall@K

$$\text{Recall@k} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \mathbb{1}(rank_i \leq k)$$

Where:
- $|Q|$ is the number of queries
- $rank_i$ is the rank of the ground truth item for the $i$-th query
- $\\mathbb{1}()$ is the indicator function, which equals 1 if the condition inside the parentheses is true, and 0 otherwise
- $k$ is the cut-off rank

Recall@k, also known as hit rate, measures the proportion of relevant items that are successfully retrieved within the top k results. In this context with one ground truth document, it's a binary measure that checks if the ground truth (correct item) is present in the top k retrieved documents.

In [10]:
def hit_rate(retrieved_docs, ground_truth, k):
    return 1.0 if ground_truth in retrieved_docs[:k] else 0.0

In [11]:
# base_model was loaded above so the corpus and query embeddings share one model.
base_model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [12]:
import numpy as np

def validate_embedding_model(validation_data, base_model, k=10):
    hit_rates = []
    reciprocal_ranks = []
    
    for data_point in validation_data:
        question = data_point['question']
        ground_truth = data_point['chunk']
        
        # Generate embedding for the question
        question_embedding = base_model.encode(question)
        
        # Retrieve documents using the embedding
        retrieved_docs = retrieve_documents_embeddings(question_embedding, k)
        
        # Calculate metrics
        hr = hit_rate(retrieved_docs, ground_truth, k)
        rr = reciprocal_rank(retrieved_docs, ground_truth, k)
        
        hit_rates.append(hr)
        reciprocal_ranks.append(rr)
    
    # Calculate average metrics
    avg_hit_rate = np.mean(hit_rates)
    avg_reciprocal_rank = np.mean(reciprocal_ranks)
    
    return {
        'average_hit_rate': avg_hit_rate,
        'average_reciprocal_rank': avg_reciprocal_rank
    }

results = validate_embedding_model(validation_data, base_model)
print(f"Average Hit Rate @10: {results['average_hit_rate']}")
print(f"Mean Reciprocal Rank @10: {results['average_reciprocal_rank']}")

Average Hit Rate @10: 0.522093023255814
Mean Reciprocal Rank @10: 0.22203303802141014


# Linear Adapter Training

![](adapter_diagram.png)

As stated in the introduction, we will be training a query-only linear adapter.

This has the added benefits of:

- Super lightweight, only one single linear transformation layer from the embedding
- Minimal added compute at run time, can be trained quickly on minimal hardware
- No need to re-embed your knowledgebase
- Proven to be almost as effective as full embedding model fine tuning

We'll be using some techniques from the [ChromaDB embedding adapters paper](https://research.trychroma.com/embedding-adapters) findings like triplet loss, random negative sampling, and some of their hyperparameters. These are outlined below:

In [13]:
import random
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.nn.utils import clip_grad_norm_

In [14]:
nvidia_loader = PyPDFLoader('data/nvidia_10k.pdf')
nvidia_pages = nvidia_loader.load()

nvidia_document = ""
for i in range(len(nvidia_pages)):
    nvidia_document += nvidia_pages[i].page_content

nvidia_chunks = text_splitter.split_text(nvidia_document)
nvidia_chunk_embeddings = base_model.encode(
    nvidia_chunks, convert_to_tensor=True, show_progress_bar=True
)

def random_negative():
    return nvidia_chunk_embeddings[random.randrange(len(nvidia_chunk_embeddings))]

random_negative()

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

tensor([-7.5518e-02, -6.5461e-02,  3.5238e-02, -6.9773e-02,  4.7480e-02,
         9.3782e-03,  6.6476e-02,  4.9236e-02,  2.3818e-02, -1.8531e-02,
         2.7387e-02, -6.6476e-02, -2.4792e-02,  1.3729e-02, -3.4187e-02,
        -3.2114e-02,  1.9466e-02,  2.6781e-02, -4.8472e-03,  1.3361e-01,
         1.5518e-01, -9.9250e-03, -2.1414e-02,  2.4773e-02, -7.5977e-03,
        -2.7638e-02,  1.6440e-02, -1.6586e-02, -2.5162e-02,  2.8020e-03,
        -6.2376e-03,  4.0461e-02, -6.3781e-02, -2.3025e-02,  7.4164e-02,
        -2.2779e-02,  6.7404e-02,  8.3820e-04,  1.2050e-02,  5.6268e-03,
        -1.3634e-02, -1.3969e-02, -5.0606e-02,  1.7115e-02,  3.3776e-02,
        -7.3527e-02,  9.1088e-02, -2.0950e-02, -9.9406e-02,  1.0682e-01,
        -2.5128e-02,  1.0068e-01, -2.6337e-02,  4.7574e-02, -2.3912e-03,
        -7.5974e-02,  5.4678e-02,  1.8240e-02, -5.5180e-02, -1.8224e-02,
        -5.9835e-02, -6.3019e-02,  9.0413e-02, -2.3001e-02, -1.6774e-02,
         3.1062e-03,  2.6292e-02, -2.7199e-02, -1.2

### Linear transformation

A linear transformation is a mathematical operation that takes an input vector and produces an output vector while preserving the operations of vector addition and scalar multiplication.

In the context of machine learning and neural networks, a linear transformation is typically represented as:
$f(x) = Wx + b$

Where:
- $x$ is the input vector
- $W$ is a matrix of weights
-  $b$ is a bias vector

Internally, `nn.Linear` creates:
- A weight matrix $W$ of shape (output_features, input_features)
- A bias vector $b$ of shape (output_features)


In [15]:
class LinearAdapter(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, input_dim)
        # Begin as an identity mapping so training refines the base model
        # instead of replacing it with a random projection.
        with torch.no_grad():
            self.linear.weight.copy_(torch.eye(input_dim))
            self.linear.bias.zero_()
    
    def forward(self, x):
        return self.linear(x)

![](tripletdataset.png)

The TripletDataset class is a custom dataset class that inherits from PyTorch's Dataset designed to work with triplet loss where each data point consists of three parts: a query, a positive example, and a negative example.

1. It retrieves the item at index idx from the training data.
2. Extracts the query and positive example from the item.
3. Uses the negative_sampler to generate a negative example.
4. Encodes the query, positive, and negative examples into embeddings using the base_model.
5. Returns the triplet of embeddings (query, positive, negative).



In [16]:
class TripletDataset(Dataset):
    def __init__(self, data, base_model, negative_sampler):
        self.data = data
        self.base_model = base_model
        self.negative_sampler = negative_sampler
        # Cache fixed query/positive vectors once; larger epoch counts then
        # train only the adapter rather than re-encoding the full dataset.
        self.query_embeddings = base_model.encode(
            [item['question'] for item in data], convert_to_tensor=True, show_progress_bar=True
        )
        self.positive_embeddings = base_model.encode(
            [item['chunk'] for item in data], convert_to_tensor=True, show_progress_bar=True
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        query_emb = self.query_embeddings[idx]
        positive_emb = self.positive_embeddings[idx]
        negative_emb = self.negative_sampler()
        return query_emb, positive_emb, negative_emb

Trainer Script
This script follows the classic machine learning optimization flow in this way:

1. Learning rate scheduler setup with warmup and decay phases
2. Initialization of LinearAdapter, loss function, optimizer, and dataloader
3. Calculation of total training steps and setup of learning rate scheduler
4. Batch generation from the TripletDataset dataloader
5. Forward pass through the LinearAdapter
6. Triplet Margin Loss calculation
7. Backpropagation and gradient clipping
8. Optimization parameter updates and learning rate adjustment
9. Epoch-wise loss reporting
10. Return of the trained adapter
Let's talk briefly about how random negative sampling and triplet loss are used:

Triplet loss is a type of loss function used in various machine learning tasks, particularly in metric learning and embedding learning. Its primary goal is to learn embeddings such that similar examples are closer together in the embedding space while dissimilar examples are farther apart.

The triplet loss operates on triplets of data points:

1. Anchor (A): The reference sample
2. Positive (P): A sample similar to the anchor
3. Negative (N): A sample dissimilar to the anchor

defined as: $L = max(d(A, P) - d(A, N) + margin, 0)$

Where:
- $d(x, y)$ is the distance function (Euclidean)
- margin is a hyperparameter that enforces a minimum distance between the positive and negative pairs

The loss encourages the model to learn embeddings where:
$$d(A, P) < d(A, N) - margin$$

This means the distance between the anchor and positive should be smaller than the distance between the anchor and negative, by at least the margin. The Negative document is dynamically randomly sampled from our NVIDIA form 10-K

In [17]:
def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return max(0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps)))
    return LambdaLR(optimizer, lr_lambda)

def train_linear_adapter(base_model, train_data, negative_sampler, num_epochs=10, batch_size=32, 
                         learning_rate=3e-4, warmup_steps=50, max_grad_norm=1.0, margin=0.2, weight_decay=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Initialize the LinearAdapter
    adapter = LinearAdapter(base_model.get_sentence_embedding_dimension()).to(device)
    
    # Define loss function and optimizer
    triplet_loss = nn.TripletMarginLoss(margin=margin, p=2)
    optimizer = AdamW(adapter.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Create dataset and dataloader
    dataset = TripletDataset(train_data, base_model, negative_sampler)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Calculate total number of training steps
    total_steps = len(dataloader) * num_epochs
    
    # Create learning rate scheduler
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    
    # Training loop
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in dataloader:
            query_emb, positive_emb, negative_emb = [x.to(device) for x in batch]
            
            # Forward pass
            adapted_query_emb = F.normalize(adapter(query_emb), p=2, dim=-1)
            positive_emb = F.normalize(positive_emb, p=2, dim=-1)
            negative_emb = F.normalize(negative_emb, p=2, dim=-1)
            
            # Compute loss
            loss = triplet_loss(adapted_query_emb, positive_emb, negative_emb)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            clip_grad_norm_(adapter.parameters(), max_grad_norm)
            
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}")
    
    return adapter

In [18]:
# Define the kwargs dictionary
adapter_kwargs = {
    'num_epochs': 30,
    'batch_size': 64,
    'learning_rate': 1e-4,
    'warmup_steps': 100,
    'max_grad_norm': 1.0,
    'margin': 0.2,
    'weight_decay': 1e-4
}

# Train the adapter using the kwargs dictionary
trained_adapter = train_linear_adapter(base_model, train_data, random_negative, **adapter_kwargs)

# Create a dictionary to store both the adapter state_dict and the kwargs
save_dict = {
    'adapter_state_dict': trained_adapter.state_dict(),
    'adapter_kwargs': adapter_kwargs
}

# Save the combined dictionary under a name that matches this run.
os.makedirs('adapters', exist_ok=True)
adapter_path = f"adapters/linear_adapter_{adapter_kwargs['num_epochs']}epochs.pth"
torch.save(save_dict, adapter_path)

/tmp/ipykernel_39316/3743545255.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  adapter = LinearAdapter(base_model.get_sentence_embedding_dimension()).to(device)


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

Batches:   0%|          | 0/108 [00:00<?, ?it/s]

Epoch 1/30, Loss: 0.0091
Epoch 2/30, Loss: 0.0042
Epoch 3/30, Loss: 0.0019
Epoch 4/30, Loss: 0.0011
Epoch 5/30, Loss: 0.0007
Epoch 6/30, Loss: 0.0004
Epoch 7/30, Loss: 0.0004
Epoch 8/30, Loss: 0.0002
Epoch 9/30, Loss: 0.0003
Epoch 10/30, Loss: 0.0003
Epoch 11/30, Loss: 0.0001
Epoch 12/30, Loss: 0.0003
Epoch 13/30, Loss: 0.0002
Epoch 14/30, Loss: 0.0002
Epoch 15/30, Loss: 0.0002
Epoch 16/30, Loss: 0.0002
Epoch 17/30, Loss: 0.0001
Epoch 18/30, Loss: 0.0001
Epoch 19/30, Loss: 0.0002
Epoch 20/30, Loss: 0.0001
Epoch 21/30, Loss: 0.0001
Epoch 22/30, Loss: 0.0001
Epoch 23/30, Loss: 0.0001
Epoch 24/30, Loss: 0.0001
Epoch 25/30, Loss: 0.0001
Epoch 26/30, Loss: 0.0001
Epoch 27/30, Loss: 0.0001
Epoch 28/30, Loss: 0.0000
Epoch 29/30, Loss: 0.0001
Epoch 30/30, Loss: 0.0000


In [19]:
# Function to encode query using the adapter
def encode_query(query, base_model, adapter):
    device = next(adapter.parameters()).device
    query_emb = base_model.encode(query, convert_to_tensor=True).to(device)
    adapted_query_emb = F.normalize(adapter(query_emb), p=2, dim=-1)
    return adapted_query_emb.cpu().detach().numpy()

In [20]:
# Later, loading and using the saved information
loaded_dict = torch.load(adapter_path, map_location='cpu')

# Recreate the adapter
loaded_adapter = LinearAdapter(base_model.get_sentence_embedding_dimension())  # Initialize with appropriate parameters
loaded_adapter.load_state_dict(loaded_dict['adapter_state_dict'])

# Access the training parameters
training_params = loaded_dict['adapter_kwargs']

print("Adapter loaded successfully.")
print("Training parameters used:")
for key, value in training_params.items():
    print(f"{key}: {value}")

Adapter loaded successfully.
Training parameters used:
num_epochs: 30
batch_size: 64
learning_rate: 0.0001
warmup_steps: 100
max_grad_norm: 1.0
margin: 0.2
weight_decay: 0.0001


/tmp/ipykernel_39316/469754793.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  loaded_adapter = LinearAdapter(base_model.get_sentence_embedding_dimension())  # Initialize with appropriate parameters


In [21]:
def evaluate_adapter(validation_data, base_model, adapter, k=10):
    hit_rates = []
    reciprocal_ranks = []
    
    for data_point in validation_data:
        question = data_point['question']
        ground_truth = data_point['chunk']
        
        # Generate embedding for the question
        question_embedding = encode_query(question, base_model, adapter)
        # Retrieve documents using the embedding
        retrieved_docs = retrieve_documents_embeddings(question_embedding, k)
        
        # Calculate metrics
        hr = hit_rate(retrieved_docs, ground_truth, k)
        rr = reciprocal_rank(retrieved_docs, ground_truth, k)
        
        hit_rates.append(hr)
        reciprocal_ranks.append(rr)
    
    # Calculate average metrics
    avg_hit_rate = np.mean(hit_rates)
    avg_reciprocal_rank = np.mean(reciprocal_ranks)
    
    return {
        'average_hit_rate': avg_hit_rate,
        'average_reciprocal_rank': avg_reciprocal_rank
    }

results = evaluate_adapter(validation_data, base_model, loaded_adapter, k=10)
print(f"Average Hit Rate @10: {results['average_hit_rate']}")
print(f"Mean Reciprocal Rank @10: {results['average_reciprocal_rank']}")

Average Hit Rate @10: 0.5197674418604651
Mean Reciprocal Rank @10: 0.22737356958287194
